# CFPB Seed v05.2 — NeMo Data Designer pipeline-only smoke

This notebook runs **10–20 engineering-only dialogues** from Seed v05.2.
The source privacy review was closed without manual verification. This
notebook therefore requires `pipeline_privacy_disposition_v01.json`,
applies additional grounding redaction before provider submission, and
marks every output:

- `provisional=true`
- `privacy_verified=false`
- `benchmark_eligible=false`

It cannot create a formal benchmark, privacy claim, prevalence estimate,
publication artifact, or distribution-calibration input.


## 0. Mount Drive, locate inputs, and create a persistent run


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").exists()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open the FinDisputeEval repository in VS Code")

os.environ["FINDISPUTEEVAL_PROJECT_ROOT"] = str(PROJECT_ROOT)
SEED_ROOT = PROJECT_ROOT / "dataset/curated/seed_pools/cfpb_dispute/seed_v052"
PRIVACY_ROOT = (
    PROJECT_ROOT
    / "dataset/curated/annotations/cfpb_seed_v05_audit"
    / "run_20260713T145423Z/privacy_qa/full_v052_v02"
)
SEED_INPUT = SEED_ROOT / "cfpb_seed_v052_generation_input.jsonl"
SEED_MANIFEST = SEED_ROOT / "seed_v052_manifest.json"
PRIVACY_DISPOSITION = PRIVACY_ROOT / "pipeline_privacy_disposition_v01.json"
for path in (SEED_INPUT, SEED_MANIFEST, PRIVACY_DISPOSITION):
    if not path.exists():
        raise FileNotFoundError(path)

SMOKE_ROOT = (
    PROJECT_ROOT
    / "outputs/generation/smoke_only/cfpb_seed_v052_pipeline_override"
)
# Set this explicitly when resuming a known run. When blank, recover
# the newest run that has raw Parquet output but no final run manifest.
RUN_ID_OVERRIDE = ""
if not RUN_ID_OVERRIDE:
    incomplete_runs = sorted(
        (
            run_root
            for run_root in SMOKE_ROOT.glob("run_*")
            if any((run_root / "raw/data_designer").rglob("*.parquet"))
            and not (run_root / "pipeline_smoke_run_manifest.json").exists()
        ),
        key=lambda path: path.name,
        reverse=True,
    )
    if incomplete_runs:
        RUN_ID_OVERRIDE = incomplete_runs[0].name.removeprefix("run_")
RUN_ID = globals().get(
    "RUN_ID",
    RUN_ID_OVERRIDE or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
)
RUN_ROOT = SMOKE_ROOT / f"run_{RUN_ID}"
PREPARED_DIR = RUN_ROOT / "prepared_inputs"
ARTIFACT_DIR = RUN_ROOT / "raw/data_designer"
VALIDATED = RUN_ROOT / "validated/dialogues.jsonl"
REJECTED = RUN_ROOT / "rejected/dialogues.jsonl"
HUMAN_REVIEW = RUN_ROOT / "review/human_review_10.jsonl"
REPORT = RUN_ROOT / "pipeline_smoke_validation_report.json"
RUN_MANIFEST = RUN_ROOT / "pipeline_smoke_run_manifest.json"
RAW_POINTER = RUN_ROOT / "raw_output_pointer.json"
for path in (PREPARED_DIR, ARTIFACT_DIR, VALIDATED.parent, REJECTED.parent, HUMAN_REVIEW.parent):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print({
    "runtime": "Colab" if IN_COLAB else "local VS Code",
    "project_root": str(PROJECT_ROOT),
    "run_root": str(RUN_ROOT),
    "resuming_incomplete_run": bool(RUN_ID_OVERRIDE),
    "seed_manifest_sha256": sha256_file(SEED_MANIFEST),
    "privacy_disposition_sha256": sha256_file(PRIVACY_DISPOSITION),
})


## 1. Install the pinned generation environment


In [ ]:
import importlib.metadata, subprocess

REQUIRED_PACKAGES = [
    "data-designer==0.7.0",
    "pydantic>=2.10,<3",
    "pandas>=2.2,<3",
    "pyarrow>=18,<23",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES])
print({
    name: importlib.metadata.version(name)
    for name in ("data-designer", "pydantic", "pandas", "pyarrow")
})


## 2. Load hash-verified embedded pipeline modules


In [ ]:
import base64, gzip

RUNTIME = Path("/content/findisputeeval_v052_pipeline_smoke") if IN_COLAB else PROJECT_ROOT / ".runtime/v052_pipeline_smoke"
RUNTIME.mkdir(parents=True, exist_ok=True)
embedded = json.loads('{"generate_multi_turn_dialogues_v052_pipeline_smoke.py": {"payload": "H4sIAAAAAAAC/7VZbXPbuBH+zl+B8kvlG4nx+fo2TtUZx5avbh3bYzk3k8lkGIiEJNQkwQNAO2rG/73PAuCb7VOvndYfEpHALvb12cUyjuMfRSU0t4Kdnt+8Y0shcvZw+PvkiIlqIyshtKw2M1UVO2ZKdS9YLnmhNo0w7FHaLbsS71USRXdbwUqVi4JtAj/T7WSZqqyoLCMuCWM3WuVNZhmvciaNwYaCr0RhmBYll1VkSIZaqwdR8SoTbh/X+N9anm392kbzsuRWZryAZKsdsxAgV4+VsVrwkj3wQubcKp1Et01lHL0WWJSZBQer2PeHs9nRIV5mSueGrSA2hz7ExynqdX5U+t7UPBNJFMdxFK21KlmarhvbaJGmTJa10qRKpSzEUZWJovBOGb+b603NtRHt3hO9aUqY44Zear+nxtGFXLVbbvDoF+yuhv3b95fSwrZFEKPe5byCCdrVd9zAGfDBlJ1LUeSdJDAET3Nh5AauSeCNtdwwDgflntF4XcJZeg2VW75nWD4Lq1P3ZIQ9hZlJ31thmsJC6yi9ub3+2+L0Lr29vr5LF1c/sTlskIjqQWpVJRthJ/H5xdXZxfLmw91i8dPJ5YgiPoiGjyCeRAx/ZIvJC94HCbypigcxOXC75Jq92OMWEFci8EjXsoDPBqQJHANPmE8/fI4OouX7678v2sNHsrxhsWps3VjzJoQ3NH/jwiSlMHmTretVSnGbIneO0lrWokDypIhhrWUu4uhscX7y4fIuXS4WZ+nNyd1fccbgQJxQa0Hi5Kms3EmVKNVrPP2xR4fJPwzO7jmf3N5dnJ9A5Fe5a/6I8I2yghvD3gtj+EZMuog5OHbG0qoQx22UfYobxGc8ZTFIpLGItfgz+LrYmiRJMmWImUzLmqwxj5e14PdCOyax90pI/GOGxNtDGcRpt4O4FfQmKL0knU9VBXMaZ/wXomeDxWNWQN5PgW0vs9tIfy9FOCmgcwVqJBup7UGn1ZuVnpWJHQuvnNlVAAvkX5pBltQ0ACS9G+u658R3Woo1C1Q4k06uFEOwIKfXO3pU2pmEAytzYbksRufXWj7wbJcCeYQJKuPsz7/+cIBpdm+YAwRd0okuumU2hSTVbChK65pwfhTloA95hLyqeJHmHhhclk0I0I5d3h2w2V/cD+8noOitJ3NIwlpg+a1hRDhTepZLgDKge8ccYxYYs1o5aEK1IUYjanaY/DE5ZCXfQShAc+Vw/MsXJNTPjbAzYm2+fGE9a+TwFnR2yysPINYw8kHRwv+K22wLwR0DJxtK10lXNtRjqIe6qVjZGMu2HDqpSjh2XvKW2vF6C9HW3s+8XMlNI+0OThQPhECMZ5mzN6ggtEXpxHaem6S1mo9yhCXVNUEIRaBGdn4FC7t9iTTeISFNXqybZr2WX5NCPQo9OYDfLZMV+xYnwXSU/wFp6FeV0+/4qefmcINLoOxPvGjEQmulJ+v4Q2WamsoHqu3YjSTOMfvWifAUwMIx8s7r1lp9SKxeZvEVsW6GKnkBzsH5Stlz1VS5l6OjOXidE6yDkHjJaaTK+TgKqUuQhlVCugjiTiHw1X10vaZesKfzhoH7jLPNBFhbsjWI3Q850D3Rm0KtJvF3nSsOSAHa13u106sQ1WR0xAH7zZx9v0exkQfjFmlDVIuvNVQx+B/ogz4Icf1aTHdtUzzito4H+r+FenAI+/ZSxKeezCsS/D/a9enwM/Bm+XF5t3hPNf79DdVnZMRH1bCM2hDRY/GoawUu2lFhMM7S3GlSZcgxYI1BsUWrg8y7Jz6GzOva2Y0msSldrfhqyeUNcZJriZCm8hDytCDMyTn1lQm7I3ngI9frUo/FK8ZXgGUH4uEwhBBVmyS6QvKjYIradcRiBP4B8RPmN6EeypLgyrId9AZcoCQ5Qeg3JLW+XJR1ISjnOMFHMUP/WuAhI+2T6KQomG9kPGStBFsA8qTZOoXcO9+E95VgLR0xR1dJe6gwqDq0QLxIXFccDd1y6l1CEdO75aXFAzyM/ANo/9FZfWZ1g6hyTQAMnrlVR1KQ9dt+P6TkJFcur2HIYH0YPui5plJokOAz5q2MbuTbt/Y3e3rCgmlWs+EintPRBndLcUv+vtJR9QtE0y1GZ2ItKiNmsprlooYmbYQMokp8zYSu7XEE6r6rTLsNadjgGA57HzD7uQHU0CXCkGI3IXzaZAXDoVVTJN4GMjw9dX1MAqp3ApniL3FkzK7r8Zckit7QE4nXOiLq8hybD7Rm4W1nhgwhpEqh01KpHCfSDiAGr7carvJee5YFbYCreseE3GxdSBiRNVaioD6625m/C42SMlgniECtE6IStalNA17tWMWh8JQ0EggTnueakgZvMmklUNhKfxfkgXbK6i3FbdWUK7yYMsqFAv9/uL00UyBUSLU3Gdd5v2u5vMK/Nxf0bw0TOZmnhE0unzlxANuZlSU1ubmTgDIViWbZxRkeL24G4uEg50oKcNoLjZSrNUN8IDdVG+eCMxXqNrUSz1vGhF2soaB/8oULYuWwCt95lIrJsPBZtqUWH1W+3LWKxiSLWweCVcbjiLsS0MWZTiMRYydiPHAuIqV09CQegemglcaxqC7FAGSYqj06u/BrNpuA26SGw9+t6AEEahorbeMywW4RD5stdW8QVKGVcVtIk0oUUD0Anav6HkGVq9dTSthSWkheC/idbAe2G2gzxSCjkijjwD7EzT/xgpo0XNanroOrcjIKOZfI0VXzwlEp3TbE0GNnt5D8Fe1baTYNsgLtLG0mrEI8gAdQp6QyTf7iOdXjKdtQGhRiQ3qpKivQQsJWLoaBopg6NNbHDMECWc4AeqRLnnEEajdUQQyBPQKahFvA4A4BOPoOMmmDjd7LfEVdDVU+lMS6hwgUdsoWFNIyzH0QiUBKzEPyFnSd7+AnH19DLJr+wrXJF5fRlSaJQly3E6UOtKlb6udGAeLfQkRFpts7KeqAhC7WoXjRVWbVyCJP/Whk4i7d/gYD3aZ+sJViosSNe+OuNHmeDAcjp470HbER2nddK/+Aqrhvb2jaw+aEvOFv/aG69X0auFwqqEFtLs3plqrRmb9qzTuRD6bdfsMBMVRIqP2wYrObg8MyvFyGd8n17dnidnE2HdwsW1mASDBJ0ZTVSAjHgrSgFa/LuJkk2J3Hr5SgeDraZzyjFEMuMe8Z3+ExOT25W/x4fftxTIGWi5eG9p6S9Gi1A82NW5g8UHtr5p9+N2V/mLI/fR5Y4+B/r+Cw1v3fVRttpr9W15hXXylxCZbXuvHOzumJwhk5637TDR960+WnKOPPI27/nZUuL98vu8z/d7ZqZ8HPzOTxIiUw4Xb+i9OeMdEgHeeD388c4Dp5auPK2s5Ht4dndvc7ni8NDRFuJcEeATH8zWOcpR4/0rDxeG/e+6PQQ6RhBH1MlxD/li4WmMDatB+j+IXvpmH0Ix6keBxRAmaOppGDptdHtMfDK/D3h+zP8+Hp9Hh0uOe6+PyG2M09AtgTSxTXfqQet8ZzNTGHfCO1Xg4t+oFlv9YOJCYtmyn7rv3ZTm/33d7jk3CmGd1uUMQBzHtnukH8sczlPY0LwrnzO92gPrtpRKru3aMnamfp0Hno/8mI2Xz0NCZMwvcLMRmHVGer5xEQ9eHsFqjotLzCq3FajvnOx4/jDBlEyRxTwsmzs6fDDS+gZCBTQs1hwXepB8dAMhnlWP+VwqXXM/VHR82Hx47KOL64pO6TC020kA/jLy7H7TjGOAeNFyfDMWma5irDB4MBgQNDHkgm8WxGhXdGDgTIOrB3yQo91hxJN38x9t/LzKHZzKFZ3POI8QUF+DmjMUS8lx4WmbXZF8QhTOk4HR3uJW8D8lfoM/rYsN9A97KehQiI2+ZyHqN04gMa6oeIn49+TIexlKvehVd0wQwJuaHx2UtPJ+4HvTKT0biP3iRdg/QfTA/HEzJ8R2ThG036S997HHwkNU2utbEvp1sExBD+tdrRVdyuC7UEGEPRD6Zel0HRG+SbWxrkxLOVEdpMn+dml1SHZDV/LByXtnDiPqMdDZtEfFBKQ7s/3/cloDvJzau9CXphKAwwkEjGhD0gdp86EEzrrgT5b7OQoINnDB96gbqRqyOL+hnn9eXF6cdjfw0ybn41pwh821082vnefI0rE973Howx6cy2uKrcpxBhI3E7ajeF6RM1EooGX3RHwsdcasOS/oMJfaFMqRnCV+M5BmVpStGdprGPQx/q0b8A76NSt5EfAAA=", "sha256": "35e4354bae683de040fe64903cfc1d5f513db8048a7c2f20969c5ae80c939386"}, "prepare_cfpb_seed_v052_pipeline_smoke.py": {"payload": "H4sIAAAAAAAC/7U8a3PbOJLf9SuwrLoqMqFoW8ns3uhG4/Mm9pRvZx1f4kzdraxiUSQkc02RHD7seDz679fdAEiApCQnezO15Yhgo9Fo9BvNtSzruuB5UHAWsIhXvNjEaVxWccjeXVz/lX3iPGIPx995E5bHOU/ilI+zNHli5Sa756wMNnnCvdHo5g4esroIOSt4woOSs7ugZEHK+Jc8icO4YnVa8IeYPwLCvIgfgvCJRXGZZ2VcxVnqMXZzF5ejXFBTsOqOF3yVFYhwVZe8ZPCwCRIWAvoiSEPusgdexKsYXgGwjgzWjVhclSNYMOIEWvBN9iAhsyJexymgSoOiCKr4gQOVIS/yyqWZJU+jktE2g1HEVzwt+ThOxxHPqzvAFAVhBbtYF1mdRnG6VrNZlRH+NU854gVC8iJDEgpvZFnWaLQqsg3z/VVd1QX3fRZv8qyoYNE0q2hCORqpsWINjCi5egZ23iXxUj3+s8xS9Tsr1a+CiyXCLEl4SAjVGu+AWDheF055FdRJFcVhJYCjoOJVvOEKUj27DP/+lqUSaR5USIECu4ZH8aJ6ypELcvwsfWo2kQM7UQxKlkej0ci//vjhv87f3fgfP3y48c+vfmEzIN7j6UNcgAiseWVbF5dX7y8/XX++OT//5exnY4bljPRHmGyPGPyHlNg93I5X8DJLHrjtEFS8Yj0YesETkFaBw1/FCZyLNtVDaUyrcv5mMXJGn87P36u1DVKOmAVsA7GvjsIazp5HRyWojp9nWVIehat86aN81hUX46BSEwt2c/nL2bv/7exGw0sDA7g1gRHIFVI/qKO4stS8ok79yfHkz8d/OXlzc/L2u7eTN/84ktrn/xocreokIVrgD9ADG/z7h7+d79phVlewg/Kole8jMgM+qopJx8RX5sIHtSsK0AFr9P784uzzzzc+cfHyCs4YFmlZCit0cLQL+XEKS3so9UkH0d/Pri4vzj/1cLVoNkEar3gpprezUcw+fLq8ufxwRZvVzgKmN/Qrdmn2BdCedJB9+HwD+wGcH5GOlouISZi0SOyhBDNwfXl9/vPl1bl/9k6u3i4nOBokSfbotybTGn08/+/Plx9hfxc/n/30CaY80yFbZGFKICpIZlVRc8sV40uwe3eboLj3Ae86XiZ8tgpA0tV7tS1pQ6Pm7RaW+un8f3xYS5CHi81pUsG9MNvkoCN2Yd0u52fjfxyPv/f8f3s9Xrz+T/UIv289fFg8T9zt7dJC6+tdOm4fh336w59uI8c+nd6+Pj2Zj73bcnHqnOKzfXobPb/Z3jqnapie5QP8fru1T3GytQcxTRnD3wn93TvldnlXVXl5Oj06uv30+vfb5ePj460HPw/TD6i/dzXcC7B1YGZZeRdMvvszGRUbreeUzIzDxj+ysiqmhC+K1yCawGJp4D0xSRqtxxg8Dk71spyntlUsLQfN6R0Y1oQLDPgf+Ea2TLLwnsUpeD5e2EmwWUbBVEKCQQsi++R48pa9YviP47KlZTkthpYWr87RBdiEz5GbBoeVqvd3/Iv4BUQaG634l8p+CJKaT3GD5kYljs42CRo8QJhF3LbqajX+d8txBpZIsiDySf+7rEwgXpmjO5vDWi76n8ViOsQ9WgU81UytM8RJSeYcl/Jw0dJGvXSIxfgLOSwmoUvBEQ/WjXPbUcf+WMAJ9GgFKcoey+kgubSPK/C0gg4iWTgeb3MfxYUtvdDsBjTchXgDcPjZPT0OC8ojSG13vy5L+SMSPLNu091yBGTiHolaQzykKNH2bOJPVG/y0gZIXKzEmCYowzieXaAtcSEiLCr/nj8Juh32mtHCkk3Sx/oqSPORfECWVYphabARkuSiTG+mzGQbcQ0hlYAlIpib4RQbZ4iQAvFaApuDrj1PghCE7fYWBq0jy2l4jj4Y1werrZANRBEI0Y4y8MTILjppl73Szq7U1KsIYogyfkFpPy+KrLBX1nstYFU8YLwMgxziVIhe2WNW3Jc50Dplz4qerWVoJK4m2UmG/Kn1u5q/sqWlaR2YJpb0jpyl8pPGW2JyVUOY3xFaF2I67z0EJhcFMHYx7a4BzNR0qLs4GSRhMHpqKXYIiQOnOFu5OuG2TD+JkQfgK0twkVNG2tHCwn7qIPGFD/VzXmAGQYBCPltImbP4TW7R4uyC9v3qAJBIVZrQIbzj4b1fQsRWlwBthUkGyH3UWYippAsOKdCR7nlLfzdxuQkqmG0yATRqynSOkpjDKBh1sqeOoc/wQo6jnCq2eqgfpd2CgmQPoWR/monJGl0A2pK2V8av5XEZgpFnkBE+NShAultsSr5hCQh4e5uEk/q1jjGYEsGov0qCdQl6PF84RKoZJu2hbZg0hZ4RXoZ5MdBWYnoDnKxTxT00YmQ15AmT1Zj1VEwaAiHPSsVnPT5b6h1s5HmrFFxuVJun6wEYoDiKMx8NxBOEGJSJlkrYIZz3wvLB0mQy5WuyIH6YpVWRJXtAwzuIP3m65gBTQq5YDgEtg/C+zhujYQiuPD1FOp1Mb0NffTgNJ+L0AbiaFU8sBkuZYjyWQO2id0Q62+k8St2FoH1bIFcFzagq6CKEryFVkVOVqrQES2cx7MF0mXA1lIaqodcgCSFfbiBvOXIBgeNVVl1gnUEwBqcYeLohJrG6dX3itTWI/bCiNgzHoE3XV9yU0tQ+k+f4GjkrvBMxV8onDkbC9oM42bsQvFS8F608RlCF4DM8WM1O8tyX5Q4/DWYdK93xO+MyXkshFhtTCvNiog9p2B9KLYWnwZInLlsVJHNKEXyM5ChEaZDZluIoWBz102Xff/cXx9WA1IaY3FBJIaTkissmx8cSXJMvjIkhACUaSBgNMg7K4edGf9kzbWdL8WiI1SsQvBa1Ln5Sn+jFXLn9PI59FCOwFNbC47/CdjLL8cCy2YfVQa0dZYQZJrEQioVPVN+D0KCGOALe6Nqykx4VXARUkBO0dCoAX0sW+iVcAeMQrKfqpdlDfOmFQwvMXyBMe4Ss0SFGiVLA1xKFhrgo4OygbhomQbxhYkUmVtxJ10BE9UeQlGNtu2LNaqxGhMI8BVA9efqNF6i5q3jtS/tueHeo5HTBULFFGYhi6DDAOpV0TbtwdKBMFKjENA+1dYgod2gVU/v+Zc/SWdeM44eIOhzLtw7bQDawl8O4zFQFHBzme5oX7KcyTUTZIV66SLq2MDCSy0Tb1Vtrb8wibzXUKkzyD2IUPJMl8lpdEWj3Kk0A1ZDZcOQPIhDZbsZQ/xJ9XZEw6NMOZkh0nG+gt+GvEZXo5FV2l4WAIUUXxMEeROjGxiciX3gzOX77DTREcURMC7G6Te7hjQuYmssvtVCnfNY4i9brqgpaFYDhw2KJDQlolG2oDD7FzYgYUhYCKHL9uhJbmwxazxru7e/PDV74TTi3VrcUJ9z7UMmvwso8Vo59cQso1tld45KFhvg3LnYlwLt7HR0o6UkDd3LMfpgRMvx3cvyiXEK/sxRzN3WJHgH+Vz1yniJavACcHLfShBEHbsphPwji96W7H/DO8LmZsmXKoxFfyG0HD0Gc4FnLJZZQHuBJIhMTUWihTas/UGgZYgfG19pVno0wbSQ4WMPDdZprJvUflsoATGXXIlQp4c62wuoYGF13NzjcPUR1+AJASKHxZqID1gYEkglz/LPwghxKmBHOFhA4ikE4lhN5ZHdn6SH000zUvZng6bBWAR34Gun53fL+mcWpjc+KtJaLxDBgIxEw3UEtUmWyVCMD9rCHCmElLFcxay4urWKIyRynw6aSJxQWDysXXtEslEbdw9OxqArfgeElEVbT+2IMh7iGMLmkIhMlHUb1aAcPVC0UV/txRksYbOlEHXSMkETEac1NMZF0qUMnWTYwzXENnR89orHs1yXM3PNs1tl0c5QQbtz3oqcGuTnBBKatv56xE930dkVUkTAoo/uFQxS4dgiHklRVSceWBL9pSPBlQ4KgA4Mp4Tj0VFykY0YN1yj2iqp7Wkm7C24iVYZdAJBnQgBN+DTToy/jVZlPBwudF+BAZ1bjHlsGV8XTtGNAgqLCewU8YvTpggc4Chxw2yGAgIE2UYa9Qy+G/Tf+RGbZZTeQa8ufrbXuCGhPOOOVoAD0BWUJmeiQc0CCfmTCUSgIGOsQj+xSMm1vgi/2sSuggXJosbEbnLRDx3H0+B/u7zDYYN2L2HYNgR/mS/wQAHmE3kb89IhvHHFtRQOIUOL26HDwlpBIcHTPKg66ex2GcK40KmJtMnti5oYX68Yw0Z8BsdCOEwjprIIFZUKic3Umx+bjk8X8ZGEy2HgFKyGLjTHB11YqwK4NYVBnNG/pW6igTfb5zGj3OzYCuS2HLp1ILq7Hs+189XM+pckLvAibX19e+u/OoNnl/dnN+cKCsQYMxWmhmxX1xiVZlEtJ5ZftBfJCRPQYCMV/pd3t0PjOGx/9xb5rIll0hxtJAxVFVP7LQjthPMT9nvg7GOepQNrMF3WaX5B1rhQaI7oRmYnV7W+DAB+ierj4tvYFk9ikxpomtSBEWyN7xTp5U4tvmJqBsoejcjFyw/vIGFpHRLPQ6BaAXDbYoXGtxW9crAmZ8CmLmina5qrLx1rMLa3/RnTewJjM7oR4glBiSclA1MvF6W2bg/em6DXCdnjf7n9qG+xogsgEo4yXsiqG5m44he2KuKuXpvddoXbVwjU1yGmyH8Cj9Sp0OdBcMBm3OcqzD1xJylBcv/mRQbc2vV76A8My6jbh+oO4dtMJqUIHA0IUIJT67Hyjbjj335bqW6mXibz3HHy/CZ58jl2LeAEAQiPb4Lp3pHQ916QHDYfHdAeFRzI/xttBurqH86F+vw6wo91rIrq9WV5P/Eq4udwEqLxqOl1p4i9Vpm4SQdVLJcVlV+eF7hqbNKpzEBTX7rFjDQ50qjuQqDMTuNrWQB+9ItC/A8ku42XmD8aUwcM+NGmXBHTmLTQ/JNo/OrUJtaCrOytX91GOcU/lQwoSm0oqYlYYnVLnb26enHrpipdaBFx6NLJ8atPrBtoSPTLirsfRpFq3Nv0SZ9cWfUNhVDa9yNbEg0klxR+4XAV9p0mTX+LWYZf8i6u6v/HaNIXbEexTtQWvtZhI2jnZoSOm6HmNEcRjR7UO6OVZbu80V1Rb6AVfrrg5amKwXTlSZ12zlGEKBQmmJLn1Hl4M3XLz4ylYmqECR5eBkDcSYbopMKSt6fOQTILfC5Mo0dlh3uvtcSgHHcseB3PA0ex0OHsdT/sSpQRPhO6DhiB4WsRgbsAQgOoEZdMlYwBtIIO7G97wGoQb686NZRyASuKo77/MQ5TmWHJdHEzzcqsfpWqhNKY+D2ydYnZpkkiRrKlUqD6wFo31pBimNSI/sIww+RJWFeinRtumfOkMLdys1ooxSS+goH93L9mzSXLh7vCQGO5rwG4wdQb34Wk7xnqtYrrsq27qfUBm63S/AeyrusX2h0O7wXf5xt0zgFnQqbqsxQkGcEdQvGS17YBBU57DKNZ2csSX9q5K+ZcXpu18uC5dQQ/CJhv6rECcJtXaFS3OVn0gQIGhfgXYxduVCpF3dD4UoAqq1sur0ek2DHCMxbotqmaY35CENQPsBYBeQPiUwGi9AgOHiVBdhSjd8lMcL80ebfU1jgfvHC8uM+oyxIqPHlEXoA6cmgx3fI+hqcHuGH/a5oNqaHEw7u9PUm8G5n69ZVAzu1+FtJN7uaU7+v8yJ43SN3vVQOd7Gz4X39aS2oamyFitNKzniL/WwCbZW4Pc04JbPcNL2g4cgDJ0ZghORg8IOzcrz3qaolDo+xtwFWXjm5rwp7cmvoHMneT2HV6pBmtoZqFiMceKK3yux8HYQpondvgfoiuH6QYNWqdBQ0BhvIFs2dchLaGlnXswkRDKz+REZVmFOoPbdlRboHZTsyt//7blzUjtW4gQnbawoBl7vNgnHvSHL/SFL/aDX+EDv87/favvazQb4vtKfJY1tMmtxnR5cdNnOn6GMNW9nUfNoSbUfh0VOtOJ30R216LtwYM5gNyl4mVneTRQ0F+FhtawGFujwKL7UfHxh4gYG3DtUxAFPPw9CEa2cP0z6X0ZsrPN0WhxVHdsuhs2qGsfVW0cv6D14VNa0BAsPquPar2rpqNCfvwBgwU46wbgrFjXmHBc0xs7gu8y4BsfFJiZ70dZCB+JajO9IIpwGZpiW+MxGswx+SHITKnXU9a6xY39rP8p5GF0baf1QYzqm8i9SKXqjvUexr2otQ8m9yIGBQCkxX5k7QeT+3dOhz1G+VT46OJRoZsc750u/Cbxb8d0/Dh2ctL5qIZw6cIj5QlL63bnYy0EoF5nDbqNandKKLUG7rq5Ma9tZojVa59dE0Zh1MAaNWwbjjslIwE8EAu1U9qAWQBn4kFbvY025NpD4YcWugioXiyjalF4oztkVA5m0bSsNRW1IlFzGsyXpP4IQLNFcKhuICyxgpt34qjFyzJlE8UgwACmV6/aOdKJd6C25uOAuTXrFsr0dnI5lGz8Mt5HVwT/jwRYBfZ9lHPflxdhQuhH/weQar24K0IAAA==", "sha256": "9b55292e058ca8ef1e586d4e61103bbcf57695ba12990ec2186ef2e5b36ed8f8"}, "validate_cfpb_v052_pipeline_smoke.py": {"payload": "H4sIAAAAAAAC/+1c63fbRnb/zr8Ciz27BmIKethJs+yqWtWWT7VNZNeS3dOVeHBAYEjCBgEED0mMqP+9vzsPYAYAKTnpfmvObkQMZu7c19y5L8S27c9BEkdBxawgjawiq/HrzbsP/25dMhZZtwffe0dWHucsiVO2l6XJ2ipX2VdmRXGQZIuald5o9DG4s94GVWC9ZWW8SFlhAUxeV1ZcWim7xfMqi+J5zCLPss7wvBYb0X53VpwuWRFXpZUEM5aUo7zIFkWwWgVVHAYJNpwX2cqqlszKC5YHBZaVhBvHl62COC0tdp8ncRhXmF2n2IBvNiZQt3EZZ2mQjPn8GUvD5Soovu6BnCRexLOEeSPbtkcjvovvz+uqLpjvW/Eqz4oKq9KsAipZWo5GaqxYAI+SqedlUC6TeKYev5RZqn5npfpVMLFFmCUJCzlAtcebrE4rVoj3JIsqXjH1Uj2PLfr3r1kq4eRBRZuqaR/wKF5U6zxOF2r8NF03eKf1Kl9bAYSSq6EcXMEA/pdHo9HI//Dx/d/P3lz5H9+/v/LPLj5bxyDBY+ltXGSpt2CVY787v3h7fvnh09XZ2efTn4wVtjvSH7HYGVn4h5BzerBdr2Blltwyx+Wz4rnVm8NfQCuYhOHP4wTS0ZZ6pBJpVV6/mo7c0eXP7//zTG1u4LJv2UIry/0Fg45yme5zZfZJr/fDeT7zSbN8aP2Rr5Tez6BPRRwxe/T27N3pp5+u/PMLUA/42maArrTTj1O+S8pW2RA8seXRgUd6knSg+j+fXpy/O7t8GnwHHh/1V0Eaz1lZcdgtaJLX+8vzq/P3F61MNObwAewBXQtKVu2HNdjDon1N+U32+EEdxZWt1hV1CoKOfjj4l8NXV4evv3999Oof+3kR3wbh2v8l2J/XSSK4cHtw1KxqKFAzo7jMszKm/TDxUBLhNmRA3c7fnl6dve0x51YasWi/tUsd9n48I3IH1hbsC87j7qWfz8/+e2Dhbczu9pc1uO6LB//woL/4w/uPA+I0xScJIMohZ5xMSfzow/m5/+H06urs48UlgDxw5tlk9hJ7AqPihdkqx5lwCvtmdn2694+Dvb94/p9e7k1f/k094veNRw/Th6Px483MHtPCc3csgOVLGJUuMOfkr3+4iVznZHLz8uTwes+7Kacn7gk9Oyc30cOrxxv3RA3zZ/mA368fnRNabKsdyjLdAp+v3MO/j/i/eyuDMCTj6MN0zVhhAuEz6B8iHYjJuRv8rTZhUEQbumVgC92b8ju8FzA2aXbjnWz+6J5g9Hryx70p/cDePwjWtFDBIvGkkEmydAElXcSVD43fTtFfxl0y4vz2dV9cxMzo4XAMXnoucVQ+zJp1dTEg5WVV5eXJZH//5vLl5mZ2d3d34+FnK9XH0eXZBR33z9C3s//6BHOi69D1SJK3jZH82ozYpmRptCmXsDqbiiWJtWIbRhfVJszSeVys3JuZ93AwfvX9o90DEZTlXQYR4C7ahLe3gAWjEldr3H+ADIW73rOm/J7jAzh4mEh2wlJi1GHyF1ykZRbimFoK3Aaq5W4X23R0+unqP95/PL/6H//NT6fnP38TH843d4xOwIsk2Vh3Mf07DNIN7vtbtrG+1GW1sYKVtcjovq0yrlE61lhK19WmrGeruNpkOUs3MdyTsooXMFabVZbGVVZs5gVjv7LNLMnCr1DcNGTJpiC/qWT4O68hBR1siFsA4FgJ94jAyKuQpFIFYbWTHbD6P59fnu1kg6R+UQdFAHGzDfRhFQOXiM1jYMyS9SZkRQXPK1m7XXOyg5+cheUyqxMck/IlRmYMf7lVkUREkmD8EGTyEc4KjABuwjDk7j6oHVru4moZp9joJnopt61LmN6y5JtHwbo80amYwhECqRY0/+j7H7jH4ZC3NeE+iGvt/ZtVVsWEbwVrAHGCi9IH9MQi6dHQztxR80j2jl3gZJOvtYTXlbBJQ8E8KywufTjDFmgunCRYzaJgImfC2wki5/Dg6LX1nUV/3LE1s223hdDi4tU53YQOh+dKfsCnTdX7JbsXv4CkIJSuGr8M5szBNVSzCTmNnEz8nSjXDH50WlaknGLWGPCgbBNNDHyXB/DG+crW7kSDC5pWLicTb8ZE4opI5YA8eiod93H7TmnuwU0timDd3+96aBdzgypLYiJ3un0Hh2bAxa6hYO5v2WS6E3vuccZhH3DLgdYJ7kPII+8KphKDq3wrjDIDUoiaHEPo/KWUc5IFEdyLEHa57Go00X9NEr2G/MYk+em0kT1X4bKez+N7L8nuoJ+udXxs2eR7/1Kzyt7FMMR4gl8i2CNiSJ19uZYj4kJGPu3uZEWMO+bYlmjardCGkAC4B1t6XGPgk0bcbXps8ekcQcR/CEXTxbFdV/O9H4ePY5cOjxhXOuSuCUroF+0tFhJyNOKBc3Gu1KwIYDKtz8T+s6LICmdun5OPbq1wbyAQtf5++f7iJwvQPghGTKwHwvPRVsfyDhcc8zlxmrTGxMdyMigwLskLXK6CFk62CI+81dcoLhwZKx1fFaRW7B4w/Owrfxy2WHdga5dn0Gh2RwQf2zfpdoMmxc2xNVgrbRonz+H8jRCYlo6pMrRvSaF4UIZxfPwuQBA4tkq4xj5siCDBtV5aHAfJMR6V++BHHVIUH3XtmckugRWfAgPeMYLut9o9cdC2LsKu2pqqWJs84ZhHEg2pbxoePAq+D1kusgse6c5bRp4T161B1X147KIoNzHxE4OSKhFqy5UtHKmPOJB+SrmZ0qnYPRSWszKdQMx0Bf4orkZWXXMzKhjteZ4yJLS+xDxcs/AjIuR2cD1fB3u/IkZ5MSX/laCqs21YsYeWcwTa4aCuAYTdW7Q9/X1ppVPX0EAxTjoYpAvmrIJ752BsJdBrvh7oWimWHcq9FJ1wXWNiTuUTPg7sbhngvtxx5Bp3QKJLOul9gVvYukB0K0pIIo1C3hrOIpkt2zXxlvMIc7V5894UnnytpEcpLmOTIkuYLWx1Q5VwJdWRUZGzjzMnsMWPSeekCLeKB/+ztR9H+vsOP8YjzhGhAZxheDftTVPsCqDNirM0kfzRabtbTEeCWAekBEVyWGcbzwYea+jxmXJic475LCQkW/Oom3m7Tr+m2V3qZ4UPXxccXvhqK2BPZ0BJBwm8epWScBwkZbIItoawgZPva4+AUTM1Lh608w98WjBkKElyOp3irSDS+oPgAeHTe9k9+pyhXpDDeke4cjgNFM+xlBTGD5d0FKLJg4DBLxupntJmgpM9M9pwX6VIFOsBeRnPyEv3efYWzjJL+CFvT6wt3jS86b4wGNd9Kdk4tKb3agDIFti9pV14jS+6jcA/k51zWhYZ94EhAxu5b6yTOUcTji24qA650HUJsbESFPzwzJDd6DKycgNWoBzzc+SST0FGTg27Yj6s6Oux9cPY+vFxO7L6fr4E4IsUDE5GGS6RebJba6VhLo8tMvE8myYWkbVXumOABn6LaikhGbdhDwAuF8ccdEfafehcrXNxC441b8vdAZAsgOJk56W0D30O4giaU5/JQkEnmRR45uFSHTdp/IctX3N1jfWbgCF9xTWoxcqwJ8M60fNWBjVUyhkg/GxG2VBNxArdOJXujcFTumCAuV2XyM7xy4nfuX+yjujSORD+RPfukRgP3VQ6n2loN+I0o+RoBwmi5jSgbJ+Gu2QzEBy8et0d/JNzhONG+kDv5aBy9Hcjx1Z5tVZnSNuLWDLpsZe0QS2Vz8o016iFoWTWNw7lOkVZDGUyPwzIYIuJtjLNIqkOvActi/F+q2mRME02yME+G3qKJW/SAfRMymx7y/4GltK+Pb3fEG1DPJEHrrXN5PAROo3/pkQDN/Faojvlv3Ef9zMBBnzpjLbeZLMNdtjiY7qtm5EGVPNDKEaKTcD1SoDKmRgWQM71ShYU4dIxqXrKUSjYgt2jSBVPHmjnx1Yj4K5LiJRMo9rwzYxyjSXZIzjyJKcbb5Hdwtsx95QZtR2mkmpKSPgxRKUhsw1fjXw/7hWJ33rIb+6i3mzfRoKA4c5RFm63sa+Jp29OUc+kstLU7gPfDhSsgl6nkfCe8yQI2TJLIlb0tgnStdORzYBSCE3S5L09h7+D1BYwkvdUSrvF9Y0zDx2G6wZeI9NQVr8bt21Z9WdhVqfBLcpXAWrvfsAr4X6YBPHqd2PVTW4/CxuUiJAsZuTeREhLyYIkT3g3znFWF/CdReiLw6sHwoZjbreVZX+B4lNKaROfHJUiV5Ge2zE5g1A7Z5cvoTp0EuRkyg18/twDptgoV+w4GALQj7SqVVs9NOO7FQDtkL8rB1VsLno4jrmLwXM22vC1rfdxCKe3tKdmZKBc84kZ3+hhFTi2xZ/vLuoEYOZC4dx3lzRBWn+X4QVanKAtetToVun3BtqDYXe7dV/qPMAuPB/Xmdj2rmybIC4b1e6CWSJJZk5rWl581fCyZWJez9BDI5T3iamrYO3zSqBP9j/xZd/Altko68NRmNXipCPfMCue2uVR5igMfdpWMe9pFQ43Zwf5ElJlNenKEV5X7b4zDoQ8PsLfx3SKCOSY241ISx9mKEB4xKNto3+pJ71nSO5JqRkZOkWHZFU3s4NLyudvhFJ+J4tkwZ0vGmJkXll6jXqXif7GbDDR3+htG1qeWmVXRWNG741quxh4wTsp+sPUFmEMb83p5ij4hGszm9ppkOElCO579UoCzTUkwEiTVBcgUoYo9rZeIf1Qt1a3WwlQhQCJC2+Tg6oG3XY70WmnptkqI/pLHZOEGiJ11X+GddlpWf4vdJNP4rUoZH3ihEdn5KNFQyrc1ARV2MczYiaNw96uLhz+26Y8nctHqP6ox5Id53draUZJRPJWhe0ozQCi5hOL99eyo8yeXtui7gtTRMk6vXBsHCh3l1Z8UO2NfCqvKFta4mCI7l0dVBIjt4tR97juxknuoEIb/bAPYqi/N85fd9tnnUA6Fu1VqoHo0N+quy+tv0uninS7WU3BRA8CJF5zv1jYHAQ00Fu1nCvr7vUFSxgFtyH+FDzDqu//DAADh6mz9Amd6YslypgwKCAHfSdD7buNl4dLQHVeUK6uuRP6rZlt+1r7TiUWHR3O2PpOf1T9mTvJoA5i2TPMy6Oir9eiRpCCt/5yrAUB8JO/lnnQxItkhslPNavb5rFTxHantQS7o05e666UuUf9wJUsad/D3uwdmrp6eGD99Zg7CoDs0u+jA5VM5CNmfkuvjPaYYpisOWdR52bQOcabfyhNeI+QClx6MLZ55GXYfwVfQxaTrB8URlrrltup8tDVImsS103dZYq+DipYaBVeLoAmY06AWxicYjVUPs/8CeKEEZRklVZU59w7ZU1ygJdHS6XJ4tgN1+i01CrizqdnCX9K+H1I0cr2bJWllY9Ow64U2GzP5KoyeHDXi8PGbQhllOHw/7Emh16aUqzq3GnDwZj5uhecDAcp3xysPDto+cbg5ZuDmG2Ya8HCZAuxZuiwA/ZTQcTvpPU30fvYH340RrSk67Prqrp6q5SBUVVtEoSiimsaLH7S1DKhfu5AClKeMqWZKsOwI1suZDQEubE+vPrL8XZ1AyRGnnkPPcfyNOlKbUPubqn6szi/z9yRV6hDijCp5IMPZpAXLcXXL7nhGiqbpzcImYHWWHLJ7c0zwq6xFJOrB17y6pNpn5HJduS+xZJW4eAZH4t2RdE60OmE7Fxjerf85kGp3wvJ2hdj68UL3EkedwmZoxzC9o7S2xdl0+f1BE5oivbIcXPPyh4sk/AmrBzrpLrd6PIbO6bEym7HhBmWSehUFhQmyMbnDXr9uZUespF4rz768dCT4Kjvfjy8c/U+Pz1sk+Hp5Fmx6XggYszRDzDjxRNlA4mZUo0gdnqS2kKlPcVpPbESCPaSUdQdQPjIAU9EV5kz5EW6XlD65L3eO11wMoaZGAGMDtzggeHzDa/t+IV6htDIDwyvNue4g2zcHolNdodhHdK5hzkZ4nOrLNocKSgjz/VF81UnugB7s/h3JwBpTsN3KkNiblc0lyB3OqTFMO26DN2N/KrxxYxOp3Yqtenys63ezd0yYvBSt6UumpZR+3Dst+lly4cdzG8TikOiNzHqLOtc5Y0Yd9JoWPV/Cok9zdlNooHQExTq6vAElY0J/6fQOKx/TxHa4LSdzEfdUohc1eT/CwO/pTDwzNxil+lUkMcM0xmxL3k7E3kDC2q2CepqmdFHRWNLFgDHVls7HVufPv4kviJGpWxNn/xgefgVH88WaHExIaPrB6IE/zFLnvasKD3rilIZvMbQJNSEAll3QZOzESkoTwvMzZx/66kI/4bn0JrJWkO1mDrcSE19OuizP+q1VLdc6/V960UZVXmgHYzua3wcDXtPmXn1mbR3gYYCnrFRbekYLKgJQk04LRZoJkirD/yNE+HTJnSWkCoc+36UhfjgV1vpBVFE2/AlLeH23h5uqj2Zkh13x+VHAm0yETf4sd0mgLRX+IKaHbeVBz3rfmyetyVL8mP7U1plNdV+Ol/B8zPQ6/MH2VU8R5bGMzg6RBtQVw7LnvD8xxpyIGEe1El1bHxBvBMaB7LXVBSehtZ8j/wEklyX9zSfZjds7YPknYDbG34nuObD4J3Ampt0Jyz1nfAToPhl9QQg+mr4CTB0fJ4CQ98PG6dOQtPPmzyClDV1Ol+B0ISmqVfObg2JnnvqFArNKuExLfXa57HWVaX71GKeOdbONT1oMdccGw+l+7m5U5B7Dna7xPSsxIJmTD/NmnciCZND+qTmZldTaECf0FhiNYHbW/1QF5RB/r0mmaRL/3EEnzqz8J+moHZ+3ydZ+76sMwrBj/4XPpuRx9BDAAA=", "sha256": "1071dde0afe5c6c5f56a3cbb228ff8ee22d23bf994e5d0e947eb6230dfbfd4d4"}}')
for name, item in embedded.items():
    raw = gzip.decompress(base64.b64decode(item["payload"]))
    if hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise ValueError(f"Embedded module hash mismatch: {name}")
    (RUNTIME / name).write_bytes(raw)
sys.path.insert(0, str(RUNTIME))
for module_name in (
    "prepare_cfpb_seed_v052_pipeline_smoke",
    "generate_multi_turn_dialogues_v052_pipeline_smoke",
    "validate_cfpb_v052_pipeline_smoke",
):
    sys.modules.pop(module_name, None)

from prepare_cfpb_seed_v052_pipeline_smoke import prepare_smoke_inputs
from generate_multi_turn_dialogues_v052_pipeline_smoke import (
    build_config,
    create_dataset,
    resolve_final_dataset_file,
)
from validate_cfpb_v052_pipeline_smoke import validate_and_route
MODULE_HASHES = {name: item["sha256"] for name, item in embedded.items()}
print(MODULE_HASHES)


## 3. Set NVIDIA_API_KEY and verify the provider alias

The key remains only in the current runtime. Use the `nvidia-text`
alias, which is backed by the OpenAI-compatible NVIDIA endpoint in
Data Designer 0.7.0.


In [ ]:
import shutil
from getpass import getpass
from data_designer.interface import DataDesigner

MODEL_ALIAS = "nvidia-text"
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY: ")
if not os.environ["NVIDIA_API_KEY"].strip():
    raise RuntimeError("NVIDIA_API_KEY is required")

def engine_aliases_and_provider_types():
    engine = DataDesigner(artifact_path=str(ARTIFACT_DIR))
    aliases = {config.alias for config in engine.get_default_model_configs()}
    provider_types = {
        provider.name: provider.provider_type
        for provider in engine.get_default_model_providers()
    }
    return engine, aliases, provider_types

engine, aliases, provider_types = engine_aliases_and_provider_types()
if MODEL_ALIAS not in aliases or provider_types.get("nvidia") != "openai":
    shutil.rmtree(Path.home() / ".data-designer", ignore_errors=True)
    engine, aliases, provider_types = engine_aliases_and_provider_types()
if MODEL_ALIAS not in aliases:
    raise RuntimeError(f"Model alias {MODEL_ALIAS!r} unavailable; aliases={sorted(aliases)}")
if provider_types.get("nvidia") != "openai":
    raise RuntimeError("NVIDIA provider_type must be 'openai'")
print({
    "model_alias": MODEL_ALIAS,
    "nvidia_provider_type": "openai",
    "provider_health_check": "deferred_to_preview",
})


## 4. Verify lineage and prepare 20 defense-in-depth redacted inputs


In [ ]:
SAMPLE_SIZE = 20
RANDOM_SEED = 20260721
SAMPLE_PATH, INPUT_MANIFEST_PATH, input_manifest = prepare_smoke_inputs(
    seed_input=SEED_INPUT,
    seed_manifest=SEED_MANIFEST,
    disposition_path=PRIVACY_DISPOSITION,
    output_dir=PREPARED_DIR,
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
)
print({
    "sample": str(SAMPLE_PATH),
    "rows": input_manifest["selected_rows"],
    "grounding_redactions": input_manifest["grounding_redactions"],
    "privacy_verified": input_manifest["policy"]["privacy_verified"],
    "benchmark_eligible": input_manifest["policy"]["benchmark_eligible"],
})


## 5. Preview 2 and generate 20 dialogues

Provider calls begin in this cell. A valid cached raw pointer avoids
accidental regeneration after reconnecting.


In [ ]:
FORCE_REGENERATE = False
RAW_OUTPUT = None
if RAW_POINTER.exists() and not FORCE_REGENERATE:
    pointer = json.loads(RAW_POINTER.read_text(encoding="utf-8"))
    candidate = resolve_final_dataset_file(Path(pointer["path"]))
    if candidate.exists() and sha256_file(candidate) == pointer["sha256"]:
        RAW_OUTPUT = candidate
        print("Generation cache hit:", RAW_OUTPUT)

# Recover a completed Data Designer dataset when generation finished
# but pointer creation was interrupted (for example, when version
# 0.7.0 returned the parquet-files directory instead of its file).
RECOVERABLE_DATASET = ARTIFACT_DIR / "dataset/parquet-files"
if (
    RAW_OUTPUT is None
    and not FORCE_REGENERATE
    and RECOVERABLE_DATASET.exists()
):
    RAW_OUTPUT = resolve_final_dataset_file(RECOVERABLE_DATASET)
    RAW_POINTER.write_text(
        json.dumps(
            {"path": str(RAW_OUTPUT), "sha256": sha256_file(RAW_OUTPUT)},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Recovered completed generation:", RAW_OUTPUT)

if RAW_OUTPUT is None and RUN_ID_OVERRIDE and not FORCE_REGENERATE:
    raise RuntimeError(
        "An incomplete run was selected but no unique raw Parquet batch "
        "could be recovered. Refusing to call the provider again."
    )

if RAW_OUTPUT is None:
    config_builder = build_config(str(SAMPLE_PATH), MODEL_ALIAS)
    results = create_dataset(
        config_builder,
        input_manifest["selected_rows"],
        ARTIFACT_DIR,
        preview_records=2,
    )
    RAW_OUTPUT = resolve_final_dataset_file(
        Path(results.artifact_storage.final_dataset_path)
    )
    RAW_POINTER.write_text(
        json.dumps(
            {"path": str(RAW_OUTPUT), "sha256": sha256_file(RAW_OUTPUT)},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Raw Data Designer output:", RAW_OUTPUT)


## 6. Route raw output into validated, rejected, and human-review artifacts


In [ ]:
report = validate_and_route(
    raw_output=RAW_OUTPUT,
    prepared_input=SAMPLE_PATH,
    input_manifest=INPUT_MANIFEST_PATH,
    disposition_path=PRIVACY_DISPOSITION,
    validated_path=VALIDATED,
    rejected_path=REJECTED,
    review_path=HUMAN_REVIEW,
    report_path=REPORT,
)
print(json.dumps({
    "pipeline_plumbing_passed": report["pipeline_plumbing_passed"],
    "raw_rows": report["raw_rows"],
    "validated_rows": report["validated_rows"],
    "rejected_rows": report["rejected_rows"],
    "rejection_reasons": report["rejection_reasons"],
    "human_review_rows": report["human_review_rows"],
    "privacy_verified": report["policy"]["privacy_verified"],
    "benchmark_eligible": report["policy"]["benchmark_eligible"],
}, ensure_ascii=False, indent=2))


## 7. Inspect the 10-row human-review pack and finalize the run manifest


In [ ]:
import pandas as pd

review_rows = [json.loads(line) for line in HUMAN_REVIEW.read_text(encoding="utf-8").splitlines() if line]
display(pd.DataFrame([
    {
        "seed_id": row.get("seed_id"),
        "product": row.get("programmatic_labels", {}).get("product"),
        "issue": row.get("programmatic_labels", {}).get("issue"),
        "passed": row.get("pipeline_smoke_validation", {}).get("passed"),
        "reasons": row.get("pipeline_smoke_validation", {}).get("reasons"),
    }
    for row in review_rows
]))

run_manifest = {
    "manifest_version": "v01",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "cfpb_seed_v052_pipeline_smoke_only",
    "run_id": RUN_ID,
    "module_hashes": MODULE_HASHES,
    "inputs": {
        "seed_manifest_sha256": sha256_file(SEED_MANIFEST),
        "privacy_disposition_sha256": sha256_file(PRIVACY_DISPOSITION),
        "prepared_input_manifest_sha256": sha256_file(INPUT_MANIFEST_PATH),
    },
    "validation_report_sha256": sha256_file(REPORT),
    "pipeline_plumbing_passed": report["pipeline_plumbing_passed"],
    "policy": report["policy"],
}
RUN_MANIFEST.write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2, sort_keys=True),
    encoding="utf-8",
)
print({
    "run_root": str(RUN_ROOT),
    "run_manifest": str(RUN_MANIFEST),
    "pipeline_plumbing_passed": report["pipeline_plumbing_passed"],
    "formal_benchmark_allowed": False,
})


## Interpretation

A successful run proves only that input preparation, provider calls,
structured generation, persistence, routing, and engineering validators
work end to end. Review the 10-row pack for naturalness and grounding.
Nothing from this run may enter the formal benchmark.
